# Mock Interview Practice — Day 25
**FAANG / Top Product Company Level — Week 4, Day 4** · Tue Nov 17, 2026

*Days 22–24 built the FAANG tier: sketches, streaming semantics, lakehouse internals. Day 25 pushes into the machinery underneath: progressive allocation math, storage-engine algorithms (Merkle trees, LSM-trees), Delta's transaction log itself, silent failures, and release engineering for data. All 18 topics are fresh vs days 1–24.*

**18 questions · 8 categories · 11 [FAANG] + 7 [HARD]**

| # | Category | Questions | Topics |
|---|----------|-----------|--------|
| 1 | SQL Advanced | Q1–Q2 | progressive tiered allocation · ratable revenue recognition |
| 2 | Python — Loops/Functions/Lists/Dicts | Q3–Q4 | Merkle tree anti-entropy · LSM-tree KV store |
| 3 | PySpark Coding | Q5–Q6 | TF-IDF cosine ranking · MinHash LSH near-duplicates |
| 4 | Spark Optimization & Deep Spark | Q7–Q8 | serialization formats (Java/Kryo/UnsafeRow) · output committers & cloud-safe commits |
| 5 | ETL — Failures, Late Data, Schema, Fault Tolerance | Q9–Q11 | GDPR right-to-erasure · Delta `_delta_log` internals · isolation levels & OCC |
| 6 | Data Modeling | Q12–Q13 | wide event table vs narrow EAV · semi-additive measures |
| 7 | Snowflake | Q14–Q15 | query performance forensics · CI/CD with git integration |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 | environment reproducibility · silent-failure tripwires · shadow/canary deploys |

### How to practice
1. Attempt each question on paper or a scratch cell **before** opening the solution. Time yourself: [FAANG] ≈ 30–40 min, [HARD] ≈ 25 min.
2. Say your reasoning OUT LOUD — interviewers grade the narrative, not just the code.
3. Executable solutions self-check with `Q# OK` prints. Talk-track questions end with a whiteboard-style answer you should be able to reproduce.

*Environment note: solutions run on serverless compute. Delta demos write to fresh scratch dirs under `/Volumes/workspace/default/etl_test_volume/` (`d25_*nb`) — safe to delete.*

## Q1 — SQL Advanced: Progressive Tiered Rewards Allocation [FAANG]

**Interviewer (Airbnb):** Our loyalty program awards points on annual spend under **progressive brackets** — exactly like income tax: the first 1000 of spend earns 1 point per unit, the next 4000 earns 2, anything above 5000 earns 3. Marketing's dashboard multiplies the whole spend by the top bracket's rate and nobody has noticed for a year.

**Requirements**

1. From `tx(user, d, amount)` compute each user's YTD spend (a window is fine).
2. Allocate spend **marginally across brackets** in ONE SQL pass — no UDFs.
3. Also compute `points_flatrate_trap = ytd_spend * top_bracket_rate` and quantify the overstatement.
4. Explain why the join condition `ytd_spend > floor_amt` performs the bracket selection, and why `LEAST(ytd, cap) - floor` is the amount taxed at each rate.

**Sample data** — three users whose totals land in different brackets: 800 / 3000 / 6500.

In [0]:
# Solution Q1 — progressive (marginal) bracket allocation
from pyspark.sql import functions as F

tx = spark.createDataFrame([
    ("u1", "2026-01-05", 300.0), ("u1", "2026-03-11", 500.0),          # total 800  -> tier 1 only
    ("u2", "2026-02-01", 1200.0), ("u2", "2026-05-20", 800.0),
    ("u2", "2026-09-30", 1000.0),                                        # total 3000 -> tiers 1+2
    ("u3", "2026-01-15", 2500.0), ("u3", "2026-06-01", 2000.0),
    ("u3", "2026-10-10", 2000.0),                                        # total 6500 -> tiers 1+2+3
], "user string, d string, amount double").withColumn("d", F.to_date("d"))
tx.createOrReplaceTempView("tx")

brackets = spark.createDataFrame([
    (1, 0.0, 1000.0, 1), (2, 1000.0, 5000.0, 2), (3, 5000.0, 1e12, 3),
], "tier int, floor_amt double, cap_amt double, rate int")
brackets.createOrReplaceTempView("brackets")

res = spark.sql("""
WITH yearly AS (
  SELECT user, SUM(amount) AS ytd_spend
  FROM tx GROUP BY user
)
SELECT y.user, y.ytd_spend,
       SUM(b.rate * (LEAST(y.ytd_spend, b.cap_amt) - b.floor_amt)) AS points_correct,
       y.ytd_spend * MAX(CASE WHEN y.ytd_spend > b.floor_amt THEN b.rate END) AS points_flatrate_trap
FROM yearly y JOIN brackets b
  ON y.ytd_spend > b.floor_amt          -- keeps every bracket the user has REACHED
GROUP BY y.user, y.ytd_spend
ORDER BY y.user
""")
res.show()

r1 = {r["user"]: (r["points_correct"], r["points_flatrate_trap"]) for r in res.collect()}
# hand-derived: u1 800*1=800 | u2 1000*1+2000*2=5000 | u3 1000+8000+1500*3=13500
assert r1["u1"][0] == 800.0 and r1["u2"][0] == 5000.0 and r1["u3"][0] == 13500.0
assert r1["u3"][1] == 19500.0  # 6500 * 3: the dashboard trap overstates by 6000 points
print("Q1 OK: marginal allocation exact; flat-rate trap overstates u3 by", r1["u3"][1] - r1["u3"][0], "points")
print("""
TALK TRACK — the two interview traps:
1) PROGRESSIVE vs FLAT: a bracket join keeps every tier the user reached
   (join on ytd > floor). The marginal amount per bracket is
   LEAST(ytd, cap) - floor  -> clipped at the bracket's top edge, floored at its bottom.
2) Retroactive-tier programs (points re-awarded when a user crosses UP mid-year)
   need the same math applied to the delta between old and new bracket boundaries —
   never re-rate the whole spend.""")

## Q2 — SQL Advanced: Ratable Revenue Recognition on a Date Spine [FAANG]

**Interviewer (Workday / Stripe):** We bill annual contracts upfront but must recognize revenue **ratably over the service period** (ASC 606). Finance's current model divides `contract_value / 12` per month and has never reconciled — month lengths differ, contracts start mid-month, and leap years bite.

**Requirements**

1. From `contracts(contract_id, start_d, end_d, value)` produce exact **monthly** recognized revenue using a **daily date spine** (`explode(sequence(...))`).
2. Daily rate = `value / total_days`; monthly revenue = sum of that month's daily rates. **Prove** the monthly allocations sum exactly to the contract value.
3. Split each contract into `recognized` (on or before a cutoff date) vs `deferred` (after it) — the P&L / balance-sheet view.
4. Explain why `value / 12` is wrong for every contract that isn't a whole 365-day year starting Jan 1.

**Sample data** — a mid-month start, a leap-February contract, and a full calendar year.

In [0]:
# Solution Q2 — daily spine recognition: exact by construction
from pyspark.sql import functions as F

contracts = spark.createDataFrame([
    ("C1", "2026-01-15", "2026-04-14", 9000.0),   # 90 days -> 100/day
    ("C2", "2024-02-01", "2024-02-29", 580.0),    # leap Feb: 29 days -> 20/day
    ("C3", "2026-01-01", "2026-12-31", 73000.0),  # 365 days -> 200/day
], "contract_id string, start_d string, end_d string, value double")
contracts = (contracts.withColumn("start_d", F.to_date("start_d"))
                      .withColumn("end_d", F.to_date("end_d")))
contracts.createOrReplaceTempView("contracts")

monthly = spark.sql("""
WITH daily AS (
  SELECT contract_id, value, day_i,
         DATEDIFF(end_d, start_d) + 1 AS total_days
  FROM contracts
  LATERAL VIEW explode(sequence(start_d, end_d)) t AS day_i
)
SELECT contract_id, trunc(day_i, 'MM') AS month,
       SUM(value / total_days) AS monthly_revenue,
       COUNT(*) AS days_active
FROM daily
GROUP BY contract_id, trunc(day_i, 'MM')
ORDER BY contract_id, month
""")
monthly.show()

rows = monthly.collect()
tot = {}
for r in rows:
    tot[r["contract_id"]] = tot.get(r["contract_id"], 0.0) + float(r["monthly_revenue"])
c1 = {str(r["month"]): float(r["monthly_revenue"]) for r in rows if r["contract_id"] == "C1"}
# hand-derived C1: Jan 17d*100=1700, Feb 28*100=2800, Mar 31*100=3100, Apr 14*100=1400
assert abs(tot["C1"] - 9000.0) < 1e-6 and abs(tot["C2"] - 580.0) < 1e-6 and abs(tot["C3"] - 73000.0) < 1e-6
assert abs(c1["2026-01-01"] - 1700.0) < 1e-6 and abs(c1["2026-02-01"] - 2800.0) < 1e-6
assert abs(c1["2026-03-01"] - 3100.0) < 1e-6 and abs(c1["2026-04-01"] - 1400.0) < 1e-6

split = spark.sql("""
WITH daily AS (
  SELECT contract_id, value, day_i, DATEDIFF(end_d, start_d) + 1 AS total_days
  FROM contracts LATERAL VIEW explode(sequence(start_d, end_d)) t AS day_i
)
SELECT contract_id,
       SUM(CASE WHEN day_i <= '2026-03-31' THEN value / total_days ELSE 0 END) AS recognized_to_q1,
       SUM(CASE WHEN day_i >  '2026-03-31' THEN value / total_days ELSE 0 END) AS deferred_remaining
FROM daily
GROUP BY contract_id
ORDER BY contract_id
""")
split.show()
targets = {"C1": 9000.0, "C2": 580.0, "C3": 73000.0}
for r in split.collect():
    assert abs(float(r["recognized_to_q1"]) + float(r["deferred_remaining"])) == targets[r["contract_id"]]
print("Q2 OK: monthly allocations sum EXACTLY to contract value; recognized + deferred = total")
print("""
TALK TRACK:
- value/12 fails every non-uniform period: 31-day Jan gets the same as 28-day Feb,
  mid-month starts are pro-rated with no month-shape awareness, and leap years drift.
- The daily spine is exact BY CONSTRUCTION (sum of value/days over all days = value);
  it also makes the P&L cutoff a trivial filter (recognized vs deferred).
- Scale note: for millions of contracts, pre-aggregate at day grain in silver and
  roll to month in gold — never recompute the spine at query time.""")

## Q3 — Python: Merkle Tree Anti-Entropy Sync [FAANG]

**Interviewer (Netflix / distributed-systems round):** Two replicas of a partitioned dataset live in different regions. Every night you must find **which rows differ** without shipping the whole partition across the WAN. This is the anti-entropy problem Dynamo and Cassandra solve with **Merkle trees**.

**Requirements**

1. Hash every record; build a binary hash tree bottom-up (parent = hash of children).
2. Compare roots — if equal, replicas are identical and you're done.
3. If roots differ, descend **only into divergent branches** until you reach the differing leaf indices.
4. Count the subtree comparisons and explain why it is O(log n) vs downloading n rows.
5. State where this pattern appears in production: Cassandra/Dynamo replica sync, Bitcoin/light-client proofs, BitTorrent piece verification, and Delta/Iceberg checksums in spirit.

**Test:** 8-record replicas where record #5 is corrupted.

In [0]:
# Solution Q3 — Merkle tree: root compare + divergent-branch descent
import hashlib

def h8(s):
    return hashlib.sha256(s.encode()).hexdigest()[:8]

def _subtree_hash(a, lo, hi):
    """Hash of leaves[lo:hi] — recursive parent = h(left || right)."""
    if hi - lo == 1:
        return a[lo]
    mid = (lo + hi) // 2
    return h8(_subtree_hash(a, lo, mid) + _subtree_hash(a, mid, hi))

def merkle_root(records):
    return _subtree_hash([h8(x) for x in records], 0, len(records))

def diff_ranges(a, b):
    """Leaf indices where two leaf-hash lists differ. Descends ONLY into
    branches whose subtree hashes disagree."""
    out, stats = [], {"comparisons": 0}
    def walk(lo, hi):
        stats["comparisons"] += 1
        if _subtree_hash(a, lo, hi) == _subtree_hash(b, lo, hi):
            return                      # identical subtree — prune the whole branch
        if hi - lo == 1:
            out.append(lo)
            return
        mid = (lo + hi) // 2
        walk(lo, mid); walk(mid, hi)
    walk(0, len(a))
    return out, stats["comparisons"]

replica_a = ["r1","r2","r3","r4","r5","r6","r7","r8"]
replica_b = ["r1","r2","r3","r4","r5","X","r7","r8"]   # leaf 5 corrupted
ha, hb = [h8(x) for x in replica_a], [h8(x) for x in replica_b]

diffs, ncomp = diff_ranges(ha, hb)
assert diffs == [5]
assert merkle_root(replica_a) != merkle_root(replica_b)

d0, c0 = diff_ranges(ha, ha)             # identical replicas: one comparison, zero descent
assert d0 == [] and c0 == 1
print(f"Q3 OK: corrupted leaf pinpointed at index {diffs} with {ncomp} subtree comparisons")
print("""
TALK TRACK:
- SYNC PROTOCOL: exchange roots only (O(1) WAN traffic). Equal -> done.
  Differ -> exchange children, recurse into mismatched halves. A single corrupted
  row in 8 requires ~log2(8)+1 comparisons instead of shipping 8 rows; for a
  million rows it's ~20 hashes per divergent range.
- PRODUCTION SIGHTINGS: Cassandra anti-entropy repair per column family;
  Dynamo partition Merkle trees; BitTorrent piece hashes; Bitcoin SPV proofs
  (a proof-of-inclusion is a path through the tree); Delta's per-commit crc
  files echo the same integrity-check idea.
- GOTCHA: rows must be CANONICALLY ORDERED + serialized identically on both
  sides, or every hash differs for non-reasons.""")

## Q4 — Python: LSM-Tree KV Store from Scratch [FAANG]

**Interviewer (RocksDB / storage-engine round, common at Uber/Databricks):** Explain and implement the engine behind most modern write-heavy stores (RocksDB, LevelDB, HBase, Cassandra's storage, and Delta's write buffers in spirit): the **LSM-tree**.

**Requirements**

1. Writes go to an in-memory **memtable**; when it fills, it is **flushed** to an immutable, sorted **SSTable**.
2. Deletes are **tombstones**, not removals — explain why a delete must also be a write.
3. Reads check memtable first, then SSTables **newest-first**; absent keys cost a scan of every level.
4. **Compaction**: merge SSTables, keep the newest version per key, and drop tombstones only when safe.
5. Discuss what production adds: per-SSTable bloom filters, leveled vs size-tiered compaction, write/read/space amplification trade-offs.

In [0]:
# Solution Q4 — a tiny LSM-tree: memtable, SSTables, tombstones, compaction
class SSTable:
    def __init__(self, pairs, level, gen):
        self.data = dict(pairs)          # key -> (seq, value); value None = tombstone
        self.level, self.gen = level, gen
    def lookup(self, k):
        return self.data.get(k)          # None = key not in THIS table

class MemTable:
    def __init__(self, threshold):
        self.d, self.threshold = {}, threshold
    def put(self, k, v, seq):
        self.d[k] = (seq, v)
    def get(self, k):
        return self.d.get(k)
    def full(self):
        return len(self.d) >= self.threshold

class LSM:
    def __init__(self, threshold=5):
        self.mem, self.ssts, self.seq, self.gen = MemTable(threshold), [], 0, 0
    def put(self, k, v):
        self.seq += 1; self.mem.put(k, v, self.seq)
        if self.mem.full(): self._flush()
    def delete(self, k):
        self.seq += 1; self.mem.put(k, None, self.seq)   # tombstone = a WRITE
        if self.mem.full(): self._flush()
    def _flush(self):
        self.gen += 1
        self.ssts.insert(0, SSTable(sorted(self.mem.d.items()), 0, self.gen))
        self.mem = MemTable(self.mem.threshold)
    def get(self, k):
        rec = self.mem.get(k)
        if rec is not None:
            return rec[1]
        for s in self.ssts:              # newest SSTable first
            rec = s.lookup(k)
            if rec is not None:
                return rec[1]
        return None                      # truly absent
    def compact(self):
        merged = {}
        for s in reversed(self.ssts):    # oldest -> newest, later writes overwrite
            merged.update(s.data)
        merged.update(self.mem.d)        # memtable is newest of all
        clean = {k: r for k, r in merged.items() if r[1] is not None}  # drop tombstones
        self.gen += 1
        self.ssts = [SSTable(sorted(clean.items()), 1, self.gen)]
        self.mem = MemTable(self.mem.threshold)
        return len(self.ssts), len(clean)

lsm = LSM(threshold=5)
for i in range(18):
    lsm.put(f"k{i}", f"v{i}")
lsm.put("k3", "v3-updated")
lsm.delete("k7")
lsm.put("k20", "v20")
assert lsm.get("k3") == "v3-updated" and lsm.get("k7") is None and lsm.get("k20") == "v20"

n_before = len(lsm.ssts)
tombstones = sum(1 for s in lsm.ssts for r in s.data.values() if r[1] is None)
n_after, n_clean = lsm.compact()
assert lsm.get("k3") == "v3-updated" and lsm.get("k7") is None
assert lsm.get("k0") == "v0" and lsm.get("k20") == "v20"
assert not any(r[1] is None for s in lsm.ssts for r in s.data.values())
print(f"Q4 OK: {n_before} SSTables -> {n_after} after compaction; {tombstones} tombstone purged; {n_clean} live keys; reads correct")
print("""
TALK TRACK — the four follow-ups they will ask:
1) WHY DELETES ARE WRITES: an older SSTable may still hold the key; only a newer
   tombstone can shadow it. Purging requires knowing no OLDER table has the key —
   hence tombstones survive until a FULL merge (or the bottom level in leveled compaction).
2) BLOOM FILTERS: per-SSTable membership sketches turn the 'scan every table'
   read path into 'usually zero disk hits'. (Day 21 built a bloom filter from scratch.)
3) AMPLIFICATION: writes are sequential (write-friendly) but compaction rewrites
   data repeatedly (write amp); reads may touch many tables (read amp); outdated
   versions linger (space amp). Leveled vs size-tiered = different trade points.
4) WHERE YOU'VE SEEN IT: RocksDB (serving + Flink state), LevelDB, HBase, Cassandra
   — and Delta's buffered writes follow the same append-immutable-file philosophy.""")

## Q5 — PySpark: TF-IDF + Cosine Similarity Search Ranking [FAANG]

**Interviewer (Google Search infra / support-search round):** Before embeddings, there was TF-IDF — and interviewers still expect you to *implement* it and know when it beats a vector DB (exact terms, tiny corpus, zero model cost, full explainability).

**Requirements**

1. From a 10-document support corpus, build a Spark ML pipeline: `Tokenizer → HashingTF → IDF → Normalizer`.
2. Vectorize the **query through the same fitted pipeline**, then rank documents by cosine similarity. Since vectors are L2-normalized, cosine = a plain **dot product** — explain why.
3. Verify the pipeline's winner against a **hand-computed TF-IDF cosine** (Python dict math, no Spark).
4. Discuss: stop words, IDF smoothing, hashing collisions (what `numFeatures` trades), why cosine over Euclidean, and when you'd graduate to BM25 or embeddings.

**Test query:** `"spark shuffle tuning"`.

In [0]:
# Solution Q5 — Spark ML TF-IDF pipeline + cosine ranking, hand-verified
from pyspark.sql import functions as F
from pyspark.ml.feature import HashingTF, IDF, Normalizer, Tokenizer
from pyspark.ml import Pipeline
import math

docs = [
    (1, "spark shuffle partition tuning and spill"),
    (2, "python list comprehension generators decorators"),
    (3, "spark shuffle memory overhead executor tuning guide"),
    (4, "sql window functions ranking running totals"),
    (5, "spark structured streaming watermarks checkpoint delta lake"),
    (6, "snowflake warehouse sizing micro partitions clustering"),
    (7, "spark tuning shuffle gc garbage collector"),
    (8, "delta lake optimize zorder file compaction vacuum"),
    (9, "kafka consumer group offset exactly once delivery"),
    (10, "pyspark dataframe groupby agg window partition"),
]
df = spark.createDataFrame(docs, "id int, text string")

pipe = Pipeline(stages=[
    Tokenizer(inputCol="text", outputCol="words"),
    HashingTF(inputCol="words", outputCol="tf", numFeatures=1 << 16),
    IDF(inputCol="tf", outputCol="tfidf_raw"),
    Normalizer(inputCol="tfidf_raw", outputCol="vec", p=2.0),
]).fit(df)
vecs = pipe.transform(df).select("id", "text", "vec")

query = "spark shuffle tuning"
qv = (pipe.transform(spark.createDataFrame([(-1, query)], "id int, text string"))
      .select("vec").collect()[0][0])

def cosine(a, b):
    return float(a.dot(b))            # both L2-normalized -> dot == cosine

scores = sorted([(cosine(r["vec"], qv), r["id"], r["text"]) for r in vecs.collect()], reverse=True)
for s, i, t in scores[:3]:
    print(f"  {s:.4f}  doc{i}: {t}")

# ---- hand computation (pure Python) as an independent check ----
def tf(tokens):
    d = {}
    for t in tokens:
        d[t] = d.get(t, 0) + 1
    return d

def idf(term):
    df_t = sum(1 for _, txt in docs if term in txt.split())
    return math.log((len(docs) + 1) / (df_t + 1)) + 1   # smoothed, Spark-style

def manual_vec(txt):
    t = tf(txt.split())
    n = math.sqrt(sum((c * idf(w)) ** 2 for w, c in t.items()))
    return {w: c * idf(w) / n for w, c in t.items()}

mv = manual_vec(query)
cos_manual = {i: sum(mv.get(w, 0) * dv.get(w, 0) for w in mv)
              for i, txt in docs for dv in [manual_vec(txt)] if True}
manual_top = max(cos_manual, key=cos_manual.get)
print("manual winner:", manual_top, "| pipeline winner:", scores[0][1])
assert scores[0][1] == manual_top
print("Q5 OK: pipeline ranking matches hand-computed TF-IDF cosine (top-1 identical)")
print("""
TALK TRACK:
- COSINE = DOT ON UNIT VECTORS: Normalizer(p=2) puts docs and query on the unit
  sphere, so similarity collapses to dot() — no magnitude bias toward long docs.
- HASHINGTF: fixed-width feature space, no vocabulary pass; collisions possible but
  rare with numFeatures >> distinct terms. CountVectorizer is the exact alternative.
- IDF SMOOTHING prevents div-by-zero on unseen terms and mimics Spark's formula.
- GRADUATION PATH: TF-IDF (exact lexical) -> BM25 (length-normalized, tunable k1/b)
  -> embeddings (semantic, needs ANN index like the IVF one from Day 22).""")

## Q6 — PySpark: MinHash LSH Near-Duplicate Detection at Scale [FAANG]

**Interviewer (Google web-scale dedup round):** You crawl 10B pages and must find near-duplicates (template spam, mirrors, syndicated articles) **without an O(n²) pairwise Jaccard** comparison. This is where MinHash + LSH live (also: the basis of Google's 1990s dedup paper, and Spark ships it).

**Requirements**

1. Shingle documents (word 3-grams), build **binary shingle-presence vectors** (`CountVectorizer(binary=True)`).
2. Fit `MinHashLSH` and use `approxSimilarityJoin` to find candidate pairs under a Jaccard-distance threshold.
3. Verify against exact Jaccard: LSH `dist == 1 - jaccard` for the pairs found.
4. **Demonstrate the threshold-recall trade-off** concretely: a 0.4 cut silently misses a true pair at distance 0.429 — show it.
5. Explain the LHS mechanics: minhash signatures approximate Jaccard; hash tables bucket signatures so only same-bucket items compare — banded locality at scale.

**Data:** 8 docs — one exact duplicate, two near-dups, four unrelated.

In [0]:
# Solution Q6 — shingles -> MinHashLSH -> approximate Jaccard joins
from pyspark.sql import functions as F
from pyspark.ml.feature import MinHashLSH, CountVectorizer

base_docs = [
    ("d1", "delta lake optimize zorder compaction vacuum retention"),
    ("d2", "delta lake optimize zorder compaction vacuum retention policy"),   # near-dup of d1
    ("d3", "structured streaming watermark state store checkpoint semantics"),
    ("d4", "structured streaming watermark state store checkpoint exactly once"), # near-dup of d3
    ("d5", "snowflake micro partitions clustering keys pruning warehouse"),
    ("d6", "kafka consumer offset exactly once delivery semantics"),
    ("d7", "delta lake optimize zorder compaction vacuum retention"),           # EXACT dup of d1
    ("d8", "python generators iterators lazy evaluation memory"),
]

def shingles(text, n=3):
    words = text.split()
    return ["_".join(words[i:i+n]) for i in range(len(words) - n + 1)]

rows = [(i, s) for i, t in base_docs for s in shingles(t)]
shdf = spark.createDataFrame(rows, "id string, shingle string")
grouped = shdf.groupBy("id").agg(F.collect_list("shingle").alias("shingles"))
cv_model = CountVectorizer(inputCol="shingles", outputCol="features", binary=True).fit(grouped)
featurized = cv_model.transform(grouped).select("id", "features")
mh_model = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=50, seed=42).fit(featurized)

def candidate_pairs(threshold):
    res = (mh_model.approxSimilarityJoin(featurized.alias("a"), featurized.alias("b"), threshold, "dist")
           .where(F.col("datasetA.id") < F.col("datasetB.id"))
           .select(F.col("datasetA.id").alias("id_a"), F.col("datasetB.id").alias("id_b"), "dist")
           .orderBy("id_a", "id_b")).collect()
    return {(r["id_a"], r["id_b"]): float(r["dist"]) for r in res}

for THRESH in (0.4, 0.5):
    print(f"threshold {THRESH}: {sorted(candidate_pairs(THRESH).items())}")

expected = {("d1","d2"), ("d1","d7"), ("d2","d7"), ("d3","d4")}
assert candidate_pairs(0.4).keys() < expected          # the 0.4 cut MISSES d3/d4
assert expected <= candidate_pairs(0.5).keys()
assert not (candidate_pairs(0.5).keys() - expected)

def jaccard(a, b):
    A, B = set(shingles(a)), set(shingles(b))
    return len(A & B) / len(A | B)

ja = jaccard(base_docs[0][1], base_docs[1][1])
assert abs((1 - ja) - candidate_pairs(0.5)[("d1", "d2")]) < 1e-9
print("Q6 OK: LSH distance == 1 - jaccard exactly; threshold 0.4 silently missed the d3/d4 pair (recall lesson)")
print("""
TALK TRACK:
- MECHANICS: a minhash signature = the k smallest hash values over the shingle
  set; P(same min) = Jaccard, so signature agreement estimates similarity.
  LSH bands signatures into buckets — only items landing in the SAME bucket are
  compared, killing the n^2 blowup (10B pages -> per-bucket work only).
- THRESHOLD TRADE-OFF: raising the cut finds more true pairs but more junk
  candidates to verify; lowering it (0.4 here) silently drops pairs just past the
  edge — 0.429 missed. Pick the threshold from a labeled sample, not intuition.
- numHashTables (bands) raises recall (and cost). 50 is comfortable for teaching;
  at web scale you tune bands/rows against your precision/recall target.""")

## Q7 — Spark Optimization: Serialization Formats — Java vs Kryo vs Tungsten UnsafeRow [HARD]

**Interviewer (Databricks Spark internals):** A job's shuffle step ships 3x more bytes than the data it reads. Walk me through **where bytes get serialized in Spark** and which format is used where.

**Requirements — whiteboard answer with configs:**

1. Three regimes: **Java serialization** (closures, RDD API, default `spark.serializer`), **Kryo** (configurable closure/shuffle object serializer), **Tungsten `UnsafeRow`** (Spark SQL's internal binary row format).
2. Why `UnsafeRow` beats both for DataFrame workloads: off-heap-capable, cache-friendly fixed-width null-bit-set layout, no per-field objects, whole-stage codegen operates directly on bytes.
3. When you genuinely still touch Java/Kryo: RDD closures, `map`/`filter` lambdas carrying big objects, accumulators, UDF payloads crossing the Python boundary (Arrow batches on Spark Connect).
4. The config stack and the off-heap decision.
5. Live: read the relevant confs off this compute and interpret what you see.

In [0]:
# Solution Q7 — serialization regimes, configs, live checks
print("""
========== Solution Q7 — three serialization regimes ==========

1) JAVA SERIALIZATION (default spark.serializer for RDD/closures)
   - Full reflection-based object graphs: class metadata ships WITH every object.
   - Safe for any Serializable class, ~5-10x bigger + slower than alternatives.
   - Used for: task closures, RDD shuffle of objects, broadcast of non-DataFrames.

2) KRYO (recommended override for RDD-heavy jobs)
   spark.serializer=org.apache.spark.serializer.KryoSerializer
   spark.kryo.registrationRequired=true     # fail fast on unregistered classes
   spark.kryoserializer.buffer.max=1g      # default 64m can overflow on big objects
   - Compact binary format; register classes to skip writing class names:
     conf.registerKryoClasses([MyClass1, MyClass2])
   - Still object-graph based — cannot operate on bytes in place.

3) TUNGSTEN UNSAFEROW (Spark SQL internal — the one that matters for DataFrames)
   - Fixed-width binary layout: [null bit set | fixed-width cols | var-length section]
   - Allocated OFF-HEAP when enabled:  spark.memory.offHeap.enabled=true
                                       spark.memory.offHeap.size=<e.g. 2g>
   - No per-field Java objects -> no GC pressure, no boxing; whole-stage codegen
     reads/writes bytes directly (see Day 24's Tungsten/codegen question).
   - This is what flows through shuffles/exchanges for DataFrame plans — you do
     NOT choose it, you avoid dropping out of it (RDD conversions, UDFs, collect).

DECISION RULE FOR INTERVIEWS:
  DataFrame job -> already UnsafeRow; keep it in Spark SQL, avoid Python UDFs.
  RDD job with closures -> Kryo + registered classes.
  Huge broadcast object -> consider a broadcast JOIN (uses the same UnsafeRow
  machinery) instead of broadcasting a Python dict.
""")

for conf in ("spark.serializer", "spark.memory.offHeap.enabled", "spark.memory.offHeap.size"):
    try:
        print(f"{conf} = {spark.conf.get(conf)}")
    except Exception as e:
        print(f"{conf} -> not settable/readable on this compute ({type(e).__name__}) — "
              "on serverless the fleet manages these; on classic clusters you own them")

import platform
from importlib.metadata import PackageNotFoundError
try:
    from importlib.metadata import version as _pkg_version
    pk = {p: _pkg_version(p) for p in ("pyspark", "delta-spark")}
except PackageNotFoundError:
    pk = {}
print("\nSpark version:", spark.version, "| Python:", platform.python_version())
print("Q7 OK: serialization stack explained; live confs interpreted (managed on serverless, tunable on classic)")

# tiny live proof that staying in Spark SQL keeps bytes binary end-to-end:
from pyspark.sql import functions as F
big = spark.range(200_000).select((F.col("id") % 1000).alias("k"), F.col("id"))
plan = big.groupBy("k").count()
_ = plan.collect()
print("note: this aggregation never materialized Java objects — UnsafeRow in, UnsafeRow out")

## Q8 — Spark Optimization: Output Committers & Cloud-Safe Commits [HARD]

**Interviewer (AWS/Azure data-platform round):** A Spark job writing to S3/ADLS occasionally produces **duplicate files or missing output** after a task retry. Explain commit protocols — why the Hadoop `FileOutputCommitter` design breaks on object stores, what the S3A "magic" committer does instead, and how Delta sidesteps the whole problem.

**Requirements — whiteboard + a runnable simulation:**

1. `FileOutputCommitter` v1: task outputs go to a temp dir; the **job committer renames** them into place at job end. Why v1 is safe but slow on cloud (thousands of driver-side renames), and why **v2** (task-direct commit) is fast but unsafe under task retries.
2. Why **rename is not atomic on S3** (copy+delete pre-list-consistency era) and what could be observed mid-crash.
3. The **S3A magic committer**: multipart-upload abort trick — zero renames.
4. **Delta's answer**: files are written invisibly; a commit is a single **atomic PUT** of one `_delta_log/NNNN.json` object at a specific version — putIfAbsent semantics. Losers re-read and retry at the next version (OCC).
5. **Simulate** the atomic-PUT race in-process: two writers, one version, one winner, one OCC retry.

In [0]:
# Solution Q8 — committer trade-offs + a live atomic-PUT/OCC simulation
print("""
========== Solution Q8 — commit protocols ==========

HADOOP FileOutputCommitter v1 (default, 'safe'):
  task -> _temporary/0/task_attempt_* ; on job success the DRIVER renames every
  task file into the final location. Correct on POSIX FS where rename is atomic.
  On S3: rename = COPY + DELETE. A 5,000-task job = 5,000 driver-side copies,
  slow + throttled; a crash mid-rename leaves the output HALF-MOVED.

FileOutputCommitter v2 (mapreduce.fileoutputcommitter.algorithm.version=2):
  tasks commit DIRECTLY (no temp dir, no job-end rename storm) -> much faster.
  UNSAFE under task failure+speculation: a retried task can produce the SAME
  file twice (duplicate/missing output) because there is no single commit point.

S3A MAGIC COMMUTTER (hadoop-aws):
  writes data as an UNCOMMITTED multipart upload; the 'commit' just completes
  the upload and writes a small marker — no rename, task-direct, still safe.
  This is why EMR/S3+Parquet jobs stopped corrupting outputs ~2019.

DELTA'S ANSWER (what you actually use on the lakehouse):
  data files are written wherever, INVISIBLE; the transaction = ONE atomic PUT
  of _delta_log/<version+1>.json. Object stores DO support atomic PUT-if-absent
  per key, so: winner commits version N; a loser that also computed off head N
  hits DELTA_CONCURRENT_MODIFICATION, re-reads, retries at N+1. No renames ever.
""")

# ---- in-process simulation of Delta's atomic-PUT commit protocol ----
committed = {}          # version -> writer  (the _delta_log)

def try_commit(writer, base_version):
    """Atomic PUT of version base_version+1. Returns False on OCC conflict."""
    version = base_version + 1
    if version in committed:        # putIfAbsent precondition fails
        return False
    committed[version] = writer
    return True

# writer B reads head=0 and commits first
assert try_commit("B", 0) is True
# writer A computed off the SAME head=0 (its snapshot is now stale) -> conflict
assert try_commit("A", 0) is False
# OCC retry: A re-reads head (now 1), re-computes, commits at 2
assert try_commit("A", 1) is True
assert sorted(committed) == [1, 2] and list(committed.values()) == ["B", "A"]
print("Q8 OK: atomic-PUT race simulated — one winner, stale writer detected, OCC retry succeeds at next version")
print("""
FOLLOW-UPS TO EXPECT:
- 'Why not just lock?' — distributed locks need a coordinator (ZooKeeper) and
  add unavailability; conditional PUT is coordination-free.
- 'Where do dead files go?' — a losing writer's data files are orphaned; Delta's
  VACUUM (default 7-day retention) reclaims them. (Retention also bounds the
  GDPR story in Q9.)
- 'Iceberg?' — same idea: atomic pointer swap of a metadata+manifest tree
  (Iceberg calls it a fast-forward commit). Hudi instead uses a timeline of
  commit instants on a lock-limited table lock when concurrent.""")

## Q9 — ETL: GDPR Right-to-Erasure Propagation Across Medallion Layers [FAANG]

**Interviewer (Uber privacy-eng round):** A user invokes Article 17 (right to erasure). Your lakehouse has bronze events, a derived silver layer, and aggregate gold tables. Legal gives you **30 days**. Deleting one row in bronze is the easy 1% — walk the whole problem.

**Requirements — implement on real Delta tables (volume-backed):**

1. Bronze with **change data feed** enabled; silver derived from it; gold = users-per-city aggregate.
2. Perform the erasure at the **source of truth**, then read the **CDF delete events** and propagate the delete to silver.
3. Gold cannot be 'un-counted' — argue why and **re-derive** it instead.
4. Write a **compliance audit record** (what was requested, what was purged, when).
5. Talk track: time travel + VACUUM retention (a DELETE is a logical tombstone — old versions still contain the PII until VACUUM), backups/replicas, ML features derived from the data, streaming consumers, and why 'anonymize instead of delete' often fails re-identification tests.

In [0]:
# Solution Q9 — CDF-propagated erasure across bronze -> silver -> gold
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/etl_test_volume"
P9 = f"{BASE}/d25_q9nb"          # fresh scratch dirs for this run

bronze_rows = [
    (42, "gdpr@x.com", "Berlin", "signup",  "2026-11-01 09:00"),
    (42, "gdpr@x.com", "Berlin", "purchase", "2026-11-02 10:00"),
    (42, "gdpr@x.com", "Berlin", "purchase", "2026-11-03 11:00"),
    (7,  "a@x.com",    "Paris",  "signup",  "2026-11-01 08:00"),
    (7,  "a@x.com",    "Paris",  "purchase", "2026-11-04 12:00"),
    (8,  "b@x.com",    "Berlin", "signup",  "2026-11-05 13:00"),
    (9,  "c@x.com",    "Lyon",   "signup",  "2026-11-06 14:00"),
]
(spark.createDataFrame(bronze_rows, "user_id int, email string, city string, event string, event_ts string")
 .withColumn("event_ts", F.to_timestamp("event_ts"))
 .write.format("delta")
 .option("delta.enableChangeDataFeed", "true")
 .mode("overwrite").save(P9))

silver0 = (spark.read.format("delta").load(P9)
           .withColumn("segment", F.when(F.col("event") == "purchase", "buyer").otherwise("browser")))
silver0.write.format("delta").mode("overwrite").save(f"{BASE}/d25_q9nb_silver")
silver0.groupBy("city").agg(F.countDistinct("user_id").alias("users")) \
       .write.format("delta").mode("overwrite").save(f"{BASE}/d25_q9nb_gold")
gold_before = {r["city"]: r["users"] for r in spark.read.format("delta").load(f"{BASE}/d25_q9nb_gold").collect()}
print("before erasure gold:", gold_before)

# 1) erase at the SOURCE OF TRUTH
spark.sql(f"DELETE FROM delta.`{P9}` WHERE user_id = 42")

# 2) read the change data feed: the delete commit's events drive downstream deletes
v_latest = spark.sql(f"DESCRIBE HISTORY delta.`{P9}`").first()["version"]
cdf = (spark.read.format("delta")
       .option("readChangeFeed", "true").option("startingVersion", str(v_latest))
       .load(P9))
del_events = [(r["_change_type"], r["user_id"]) for r in cdf.collect()]
print("CDF delete events:", del_events)
assert all(t == "delete" and u == 42 for t, u in del_events) and len(del_events) == 3

spark.sql(f"DELETE FROM delta.`{BASE}/d25_q9nb_silver` WHERE user_id = 42")

# 3) gold: RE-DERIVE — an aggregate cannot be reliably 'un-counted'
spark.read.format("delta").load(f"{BASE}/d25_q9nb_silver") \
    .groupBy("city").agg(F.countDistinct("user_id").alias("users")) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{BASE}/d25_q9nb_gold")

for nm in ("d25_q9nb", "d25_q9nb_silver"):
    assert spark.read.format("delta").load(f"{BASE}/{nm}").where("user_id = 42").count() == 0
assert spark.read.format("delta").load(P9).count() == 4
gold_after = {r["city"]: r["users"] for r in spark.read.format("delta").load(f"{BASE}/d25_q9nb_gold").collect()}
assert gold_after == {"Paris": 1, "Berlin": 1, "Lyon": 1}, gold_after

# 4) compliance audit record (a real deployment appends this to a governed table)
audit = [("REQ-1001", 42, "2026-11-17",
          ["bronze", "silver", "gold"], 3)]
print("audit record:", audit)
print("Q9 OK: erasure propagated via CDF; gold re-derived (Berlin 2->1); user 42 gone from bronze+silver")
print("""
TALK TRACK — the 30-day reality:
- DELETE IS A TOMBSTONE: time travel still exposes pre-erasure versions until
  VACUUM (log retention + expired-snapshot window, default 7 days). A lawful
  erasure SLA usually means: DELETE now, VACUUM on a schedule that fits 30 days.
- CASCADES: ML features trained on the user's data, downstream exports, replica
  environments, backups (regulators accept backup expiry windows — document them),
  streaming consumers (send tombstones), reverse-ETL sinks (Day 18).
- AGGREGATES: re-derive, don't patch — count fixes propagate only by recomputation.
- ANONYMIZATION is NOT erasure: keyed pseudonyms re-identify via joins; only
  true removal (or provable k-anonymous aggregation) satisfies Article 17.""")

## Q10 — ETL: Delta `_delta_log` Internals — Reading the Transaction Log Itself [FAANG]

**Interviewer (Netflix lakehouse-platform round):** Everyone uses `DESCRIBE HISTORY`. Today you open the **actual `_delta_log`**: what files exist, what a commit contains, how a snapshot is rebuilt, and what checkpoints are for.

**Requirements — live on a volume-backed Delta table:**

1. Create a Delta table at a **path** and make ~14 commits (creates, appends, UPDATE, DELETE).
2. List `_delta_log/` — identify commit JSONs (JSONL), `.crc` sidecars, `_staged_commits/` (serverless), and any checkpoint parquet.
3. Parse a raw commit: the **txn / add / remove / metaData / commitInfo** actions. Explain that a commit file is **newline-delimited JSON — one action per line**.
4. Reconcile: number of JSON commits == `DESCRIBE HISTORY` rows.
5. Explain **checkpoints** (every `delta.checkpointInterval` commits, default 10, the full table state is parquet'd so a reader doesn't replay thousands of JSONs) and note what this serverless compute shows instead.

In [0]:
# Solution Q10 — inspect the real _delta_log of a path-backed Delta table
import json

P10 = "/Volumes/workspace/default/etl_test_volume/d25_q10nb"

spark.createDataFrame([(0, "init")], "k int, v string") \
     .write.format("delta").mode("overwrite").save(P10)
for i in range(1, 12):                      # 11 appends -> version 11
    spark.createDataFrame([(i, f"v{i}")], "k int, v string") \
         .write.format("delta").mode("append").save(P10)
spark.sql(f"UPDATE delta.`{P10}` SET v = 'upd' WHERE k = 1")   # version 12
spark.sql(f"DELETE FROM delta.`{P10}` WHERE k = 2")           # version 13

files = [f.name for f in dbutils.fs.ls(P10 + "/_delta_log")]
json_commits = sorted(f for f in files if f.endswith(".json"))
crc = [f for f in files if f.endswith(".crc")]
chk = [f for f in files if "checkpoint" in f]
staged = [f for f in files if "_staged" in f]
print("log entries:", files)

hist = spark.sql(f"DESCRIBE HISTORY delta.`{P10}`").orderBy("version")
h_ops = [(r["version"], r["operation"]) for r in hist.collect()]
print("history:", h_ops)
assert len(json_commits) == len(h_ops), "every commit JSON == one history row"
print(f"{len(json_commits)} commit JSONs == {len(h_ops)} history rows; crc sidecars: {len(crc)}")

# a commit file is JSONL: ONE ACTION PER LINE, not one JSON object
last = json_commits[-1]
lines = [json.loads(l) for l in dbutils.fs.head(P10 + "/_delta_log/" + last).splitlines() if l.strip()]
kinds = {}
for l in lines:
    k = next(iter(l.keys()))
    kinds[k] = kinds.get(k, 0) + 1
print("actions in commit", last, "->", kinds)
assert "commitInfo" in kinds and ("remove" in kinds or "add" in kinds)

if chk:
    cp = spark.read.parquet(P10 + "/_delta_log/" + chk[0])
    print("checkpoint parquet rows:", cp.count(), "(the full table state in one file)")
else:
    print("no checkpoint parquet visible on this compute —", "_staged_commits/" if staged else "plain log",
          "(serverless writes stage commits then finalize; on classic compute a checkpoint parquet appears every delta.checkpointInterval=10 commits)")
print("Q10 OK: raw log parsed (JSONL actions), commit count reconciled with history, log anatomy explained")
print("""
TALK TRACK — how a reader builds a snapshot:
1. List _delta_log, find the latest checkpoint parquet (say v20) -> full file set state.
2. Replay ONLY the commit JSONs after it (v21, v22, ...) applying add/remove actions.
3. That snapshot = the table. DESCRIBE HISTORY reads the same log; time travel =
   replay up to version N. CDF (_change_data files) is the per-commit diff of it.
4. crc files are checksums per commit (integrity, echoes Q3's Merkle idea).
5. STAGED COMMITS (this compute): serverless writes stage log entries out-of-order
   under _staged_commits/ and finalize them into position — an object-store-safe
   write path that never blocks on rename (ties directly into Q8).""")

## Q11 — ETL: Delta Isolation Levels & Optimistic Concurrency [HARD]

**Interviewer (concurrency round — a step up from Day 14's OCC retry):** Two schedulers run the same MERGE job simultaneously every hour (misconfigured trigger) and sometimes one fails with `DELTA_CONCURRENT_MODIFICATION`. Explain **why it happens, when it CAN'T, and how the isolation level changes the answer.**

**Requirements:**

1. Live proof that **blind appends never conflict** (two writers, same table, both succeed) — and why (appends don't read the files they write beyond the log).
2. Why a MERGE that **rewrites matched files** can conflict: it read file X in its snapshot; another writer already replaced X; committing would violate the snapshot assumption.
3. `delta.isolationLevel`: **`WriteSerializable` (default)** vs **`Serializable`** — what changes (which conflicts are accepted) and the guidance for choosing.
4. The production retry wrapper: catch the OCC exception, **exponential backoff + jitter**, re-run — and prove the wrapper converges (simulate the injected conflict).
5. One live MERGE on a real path-backed table to anchor the mechanics.

In [0]:
# Solution Q11 — conflicts that can't happen, conflicts that can, retries that converge
import random, time

P11 = "/Volumes/workspace/default/etl_test_volume/d25_q11nb"
spark.createDataFrame([(1, "a", 10), (2, "b", 20)], "k int, name string, amt int") \
     .write.format("delta").mode("overwrite").save(P11)

# 1) BLIND APPENDS NEVER CONFLICT — both writers win (snapshot isolation for appends)
spark.createDataFrame([(3, "c", 30)], "k int, name string, amt int").write.format("delta").mode("append").save(P11)
spark.createDataFrame([(4, "d", 40)], "k int, name string, amt int").write.format("delta").mode("append").save(P11)
n = spark.read.format("delta").load(P11).count()
assert n == 4
print(f"1) two concurrent-style appends both committed: table now has {n} rows — no conflict possible")

# 2) a real MERGE — the operation class that CAN conflict (it reads files it rewrites)
src = spark.createDataFrame([(2, "b", 99)], "k int, name string, amt int")
src.createOrReplaceTempView("d25_q11_src")
spark.sql(f"""
MERGE INTO delta.`{P11}` t USING d25_q11_src s
ON t.k = s.k
WHEN MATCHED THEN UPDATE SET t.amt = s.amt
WHEN NOT MATCHED THEN INSERT *
""")
assert spark.read.format("delta").load(P11).where("k = 2").first()["amt"] == 99
print("2) MERGE committed: k=2 -> amt 99 (this is the op that conflicts when a peer rewrote the same files first)")

# 3) isolation levels
print("""
3) delta.isolationLevel:
   WriteSerializable (DEFAULT) — serializable for WRITES, but permits some
     concurrent non-conflicting file operations; the pragmatic lakehouse default.
   Serializable — strictest serializability; rejects additional concurrent
     operations (e.g. blind appends into files a writer is rewriting),
     trading conflict rate for the strongest guarantees.
   Neither level ever produces corrupt output — a conflict is REJECTED with
   DELTA_CONCURRENT_MODIFICATION, never half-applied. That's the OCC contract.
""")

# 4) production retry wrapper with backoff+jitter (conflict INJECTED on attempt 1)
def merge_with_retry(path, attempt_fn, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            if attempt == 1:
                raise Exception("DELTA_CONCURRENT_MODIFICATION: another writer committed first")
            return attempt_fn()               # re-read latest snapshot inside the fn
        except Exception as e:
            if "CONCURRENT" not in str(e) or attempt == max_retries:
                raise
            backoff = min(2 ** attempt, 30) / 1000 + random.random() / 1000
            print(f"   attempt {attempt} conflicted -> retrying in {backoff*1000:.1f} ms (backoff + jitter)")
            time.sleep(backoff)

result = merge_with_retry(P11, lambda: spark.read.format("delta").load(path := P11).count())
print(f"4) retry wrapper converged; re-read gives {result} rows")
assert result == 4
print("Q11 OK: appends conflict-free, MERGE is the conflict-prone class, isolation levels explained, retry wrapper proven")

## Q12 — Data Modeling: Wide Event Table vs Narrow EAV — Pivoting Back [HARD]

**Interviewer (healthcare/IoT modeling round):** Our vendor feeds 3,000 lab-result attributes as an **entity-attribute-value (EAV)** river — one row per (patient, visit, attribute, value). Analytics keeps asking for `AVG(hemoglobin)`. When is EAV right, when is a wide table right, and how do you pivot between them at scale?

**Requirements:**

1. Model the same 3 events both ways: wide event table (fixed columns) vs narrow EAV (attr/value rows). Count the **row explosion**.
2. Answer the same analytical question off both models with the same result; note everything EAV forces you to add (groupBy-pivot, casts, first/last aggregates).
3. Modeling decision rules: schema churn vs query patterns, storage, typing (EAV values become strings), BI-friendliness.
4. The scale story: `pivot` needs the values list upfront (collect) — why that breaks at 3,000 attrs, and the alternative (explicit key-value + join at query time, or a model per attribute family).

In [0]:
# Solution Q12 — same facts, two shapes; the pivot tax made explicit
from pyspark.sql import functions as F

# WIDE: one row per event, typed columns
events_wide = spark.createDataFrame([
    ("e1", "u1", "page_view", 100, 0, 0.0, "ios", None),
    ("e2", "u2", "purchase", 0, 1, 59.99, "android", "card"),
    ("e3", "u1", "page_view", 200, 0, 0.0, "ios", None),
], "event_id string, user_id string, event_type string, view_ms int, items int, order_value double, platform string, pay_method string")

# NARROW EAV: one row per (event, attribute) — same facts, exploded
wide_rows = events_wide.collect()
eav_rows = []
for r in wide_rows:
    d = r.asDict()
    for col in ("view_ms", "items", "order_value", "platform", "pay_method"):
        if d[col] is not None:
            eav_rows.append((d["event_id"], d["user_id"], d["event_type"], col, str(d[col])))
eav = spark.createDataFrame(eav_rows, "event_id string, user_id string, event_type string, attr string, val string")
print(f"wide rows: {events_wide.count()} | EAV rows: {eav.count()} (3 events -> {eav.count()} attribute rows: the row explosion)")

# same question both ways: total order_value on purchases
wide_res = (events_wide.where("event_type='purchase' AND order_value IS NOT NULL")
           .agg(F.sum("order_value").alias("total")).collect()[0]["total"])

eav_res = (eav.groupBy("event_id", "user_id", "event_type")
    .pivot("attr", ["view_ms", "items", "order_value", "platform", "pay_method"])
    .agg(F.first("val"))
    .where("event_type = 'purchase'")
    .agg(F.sum(F.col("order_value").cast("double")).alias("total")).collect()[0]["total"])
assert float(wide_res) == float(eav_res) == 59.99
print("Q12 OK: both shapes answer identically (59.99), but EAV paid groupBy+pivot+cast to get there")
print("""
TALK TRACK — decision rules:
- EAV WINS when: schema churns weekly (new lab analytes), attributes are sparse
  (a patient has 20 of 3000), or ingestion must stay schema-flexible.
- WIDE WINS when: attributes are dense, BI runs ad-hoc typed filters/aggregates,
  and downstream expects stable columns. Storage: EAV stores (key strings per
  value); wide stores nulls — crossover depends on sparsity.
- THE TAXES: EAV kills typing (values become strings -> cast everywhere),
  multi-attribute filters become pivots/joins, and SQL gets unreadable.
- SCALE: pivot() wants the attr list UPFRONT (driver collect). At 3,000 attrs,
  don't pivot — either model per attribute FAMILY (labs/vitals/meds wide-ish
  tables) or answer in place: filter attr = 'hemoglobin' and cast only that column.
- HYBRID REALITY: bronze EAV (ingest anything) -> curated wide per use case.
""")

## Q13 — Data Modeling: Semi-Additive Measures in Snapshot Facts [FAANG]

**Interviewer (fintech modeling round):** Our nightly `balance_snapshot` (account, date, balance) feeds the CFO dashboard. Someone wrote `SUM(balance)` grouped by quarter and reported total 'holdings' of 970 — the finance team says the right number is 340. Diagnose the measure and fix the aggregation.

**Requirements:**

1. Classify measures: additive, **semi-additive**, non-additive — give one example of each from a bank's data.
2. Show the two WRONG aggregations: `SUM` across time (double-counts), and `AVG` of per-snapshot totals without thinking about it.
3. Produce the two CORRECT answers: **end-of-period balance** (last snapshot in the window, per account) and **average daily total balance** (sum within a day, then average across days).
4. Explain the Kimball rule (never SUM a semi-additive across time; pick an endpoint or average) and where the same trap reappears: inventory, headcount, temperature, FX rates.

In [0]:
# Solution Q13 — the classic balance-snapshot trap
from pyspark.sql import functions as F

bal = spark.createDataFrame([
    ("2026-01-31", "acct_a", 100.0), ("2026-01-31", "acct_b", 200.0),
    ("2026-02-28", "acct_a", 150.0), ("2026-02-28", "acct_b", 180.0),
    ("2026-03-31", "acct_a", 120.0), ("2026-03-31", "acct_b", 220.0),
], "snap_date string, account string, balance double").withColumn("snap_date", F.to_date("snap_date"))
bal.createOrReplaceTempView("bal")

wrong = spark.sql("SELECT SUM(balance) s FROM bal").collect()[0]["s"]
print(f"WRONG: SUM(balance) across time = {wrong} (counts every account 3x)")

# correct 1: average DAILY TOTAL balance (sum within day, then avg across days)
right_avg = spark.sql("""
WITH daily AS (SELECT snap_date, SUM(balance) AS total FROM bal GROUP BY snap_date)
SELECT AVG(total) AS avg_daily_total FROM daily
""").collect()[0]["avg_daily_total"]
print(f"correct avg-daily-total balance = {float(right_avg):.2f}  (Jan 300, Feb 330, Mar 340 -> 970/3)")

# correct 2: END-OF-PERIOD balance (the balance-sheet view)
eopd = {r["account"]: r["balance"] for r in spark.sql("""
SELECT account, balance FROM (
  SELECT account, balance,
         ROW_NUMBER() OVER (PARTITION BY account ORDER BY snap_date DESC) rn
  FROM bal)
WHERE rn = 1 ORDER BY account
""").collect()}
total_eop = sum(eopd.values())
print(f"correct end-of-period total = {total_eop} ({eopd})")

assert wrong == 970.0 and abs(float(right_avg) - 970.0 / 3) < 1e-9
assert eopd == {"acct_a": 120.0, "acct_b": 220.0} and total_eop == 340.0
print("Q13 OK: SUM-across-time 970 rejected; avg-daily 323.33 and end-of-period 340 produced")
print("""
TALK TRACK:
- CLASSIFICATION: additive (transaction amount: sums over ANY dimension),
  semi-additive (balance/inventory/headcount: sums over account/product but NOT
  over time), non-additive (rate/ratio/temperature: never sums, e.g. FX rate).
- KIMBALL RULE: for semi-additives pick an ENDPOINT (end-of-period for 'as-of'
  reporting) or an AVERAGE-of-totals (for 'typical level' questions) — never SUM
  across the time dimension, and never AVG account-level balances into an
  'average account' unless that's literally the question.
- WHERE IT BITES AGAIN: inventory on-hand, employee headcount, sensor levels,
  exchange rates — any periodic snapshot fact. If the source is event-based,
  consider deriving a date-spine snapshot (Day 23) instead of trusting cadence.""")

## Q14 — Snowflake: Query Performance Forensics [HARD]

**Interviewer (Snowflake PS / platform round):** A 9 a.m. dashboard query that ran 40s for months now runs 9 minutes and spills. You have `EXPLAIN`, the query profile, and `QUERY_HISTORY`/`QUERY_OPERATOR_STATS`. Diagnose systematically and propose the fix — including when NOT to add a cluster key.

**Requirements — a forensics ORDER OF OPERATIONS:**

1. `EXPLAIN <query>` first: check **partition pruning** (partitions accessed vs total), join order, and whether the filter that used to prune is still sargable (function-wrapped, type-cast).
2. `QUERY_HISTORY` columns that matter: `partitions_scanned/partitions_total`, `bytes_spilled_to_local/remote_storage`, `total_elapsed_time`, and grouping by `warehouse_size` to separate data growth from compute regression.

3. The **spill ladder**: local spill (fast SSD) → remote spill (object store — 10x worse). Fixes in order: is the join producing a cross-product? can the query be rewritten (pre-aggregate, eliminate the join)? only then resize the warehouse.
4. When clustering actually helps vs when **materialized/dynamic tables, search optimization, or a better model** is the right tool — and how your answer stays distinct from Day 15/24's accelerator questions.

Snowflake SQL examples are shown as strings (no Snowflake attached to this notebook).

In [0]:
# Solution Q14 — the forensics runbook (Snowflake SQL as reference strings)
q_explain = "EXPLAIN SELECT ... FROM events WHERE user_id = 42 AND event_date >= '2026-10-01'"
q_history = """
SELECT query_id, total_elapsed_time, warehouse_size,
       partitions_scanned, partitions_total,
       bytes_spilled_to_local_storage AS local_spill,
       bytes_spilled_to_remote_storage AS remote_spill
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE '%dashboard_query%'
  AND start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
ORDER BY start_time
"""
q_op_stats = """
SELECT operator_id, operator_type,
       partitions_assigned, partitions_total,
       bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage
FROM TABLE(GET_QUERY_OPERATOR_STATS('<query_id>'))
ORDER BY operator_id
"""
print("""
========== Solution Q14 — order of operations ==========

STEP 0 — DIFF THE ENVIRONMENT, NOT JUST THE QUERY
  Same query text? Same warehouse size? partition_depth growing? Group
  QUERY_HISTORY by day: did elapsed time drift with data volume (clustering
  candidate) or JUMP on a code deploy (regression candidate)?

STEP 1 — EXPLAIN: is the big filter still PRUNING?
  partitions_accessed ~= partitions_total means NO pruning.
  Usual suspects: filter wrapped in a function
    WHERE UPPER(email) = 'X'      -- kills micro-partition zone-map pruning
    WHERE event_date::string LIKE '2026-10%'  -- cast defeats min/max pruning
  Fix: persistent-computed columns or sargable rewrites
    WHERE email = 'x' AND event_date >= ... AND event_date < ...

STEP 2 — QUERY_OPERATOR_STATS: find the SPILLING JOIN
  A hash join whose build side exceeds warehouse memory:
    local spill  (SSD) = degraded but alive;
    REMOTE spill (S3)  = the 10x cliff. remote_spill > 0 is the smoking gun.

STEP 3 — FIX LADDER (cheapest first):
  a) REWRITE: pre-aggregate the build side, eliminate the join (denormalize
     the dashboard's 2 columns into the fact — the Day 19 star-schema answer),
     push filters before the join.
  b) PRUNE: natural-key design — if the table arrived by load order and users
     filter on event_date, co-locate it: CLUSTER BY (event_date, user_id).
     Check SYSTEM$CLUSTERING_INFORMATION first; a monotonically increasing
     load-order table usually clusters WELL on date already.
  c) RESIZE: warehouse up = more memory + spill headroom. It MASKS the
     problem — acceptable as a stopgap; costs scale with it.

STEP 4 — ACCELERATORS, ONLY WHEN THE PATTERN IS KNOWN (distinct from Day 15/24):
  SEARCH OPTIMIZATION: point-lookup equality predicates (WHERE user_id = 42)
    — not range scans, not selective joins of unknown shape.
  MATERIALIZED/DYNAMIC TABLE: when the dashboard recomputes the same window
    every refresh — precompute once per interval (dynamic = auto-refresh).
  CLUSTERING: when filters are known AND recurring AND pruning is currently bad.
  Rule for interviews: pick the accelerator that matches the ACCESS PATTERN,
  not the one that's trendiest.
""")
for probe in ("partitions_scanned", "partitions_total", "bytes_spilled_to_remote_storage"):
    assert probe in q_history, probe
assert "GET_QUERY_OPERATOR_STATS" in q_op_stats
print("Q14 OK: runbook covers environment diff -> EXPLAIN pruning -> operator spill -> fix ladder -> accelerator choice")

## Q15 — Snowflake: CI/CD with Git Integration & Blue-Green Schema Swaps [HARD]

**Interviewer (Snowflake DevOps round):** Analysts' dashboards must never see a half-deployed schema change. Design a **zero-downtime deployment pipeline** for Snowflake objects: DDL in git, review via PR, atomic swap into production.

**Requirements — the design you must defend:**

1. **Git integration**: `CREATE GIT REPOSITORY` + `FETCH` + `EXECUTE IMMEDIATE FROM @repo/branch/path/file.sql` — deploy straight from a branch.
2. **Blue-green swap**: deploy the new shape to a shadow schema, backfill/copy, then `ALTER TABLE prod SWAP WITH staging` (metadata-only, instant, atomic).
3. Why `CREATE OR ALTER` (and forward-only, numbered migrations) beats `DROP+CREATE`, plus the rename/view-indirection alternative.
4. The traps: grants don't travel with swaps, views referencing old tables, pipelines/tasks touching the shadow, and testing the DEPLOY in a clone environment (`CREATE SCHEMA ... CLONE`) first.

Scripts shown as strings — narrate the pipeline stages.

In [0]:
# Solution Q15 — the zero-downtime deploy pipeline, stage by stage
setup_git = """
CREATE OR REPLACE GIT REPOSITORY analytics_repo
  ORIGIN = 'https://github.com/org/analytics.git'
  CREDENTIALS = (TYPE = PASSWORD) -- stored in a SECRET, never inline
  MAIN_BRANCH = 'main';
"""
stage_fetch = """
ALTER GIT REPOSITORY analytics_repo FETCH;
-- CI validates the PR here: SHOW on the staged SQL, lint, clone-test (below)
"""
deploy_script = """
EXECUTE IMMEDIATE FROM
  @analytics_repo/branches/main/migrations/0042_add_segment_to_customers.sql;
"""
blue_green = """
-- BLUE (current prod) stays live the whole time
CREATE SCHEMA IF NOT EXISTS deploy_v2;
CREATE OR ALTER TABLE deploy_v2.customers (
  customer_id NUMBER, segment VARCHAR, ...  -- new shape
);
-- backfill / copy from blue (CTAS or INSERT ... SELECT, validated row counts)
INSERT INTO deploy_v2.customers SELECT ..., NULL FROM prod.customers;
-- GREEN: metadata-only instant atomic swap
ALTER TABLE prod.customers SWAP WITH deploy_v2.customers;
-- old shape now sits in deploy_v2: instant rollback = swap back
"""
clone_test = """
CREATE SCHEMA prod_staging CLONE prod;   -- zero-copy test bed
-- run migrations against prod_staging; row-count + EXPLAIN-diff gates;
-- only after green does the swap sequence run against prod
"""
print("""
========== Solution Q15 — pipeline stages ==========
PR -> CI:  FETCH repo; clone prod into prod_staging; EXECUTE IMMEDIATE the
           migration against the CLONE; gates (row counts, EXPLAIN diffs,
           smoke queries from the dashboard service account).
DEPLOY:    blue-green above. SWAP is metadata-only: instant, atomic, reversible.
           Dashboards hold a connection to table NAMES, which never change.

WHY NOT DROP+CREATE: locks/visibility gaps, grants lost, time-travel severed.
CREATE OR ALTER: converges an object to the declared shape (adds columns,
  converges properties) without dropping it — idempotent re-runs.
Numbered forward-only migrations keep history linear: never edit an applied
  migration; ship a corrective one.

TRAPS TO NARRATE (interviewers fish for these):
1) GRANTS: SWAP moves the TABLE, not the PRIVILEGES on the old object — re-grant
   on the shadow BEFORE the swap (or use future grants / role-based defaults).
2) VIEWS/UDFs referencing the table keep resolving (names stable) — but a
   view with SELECT * captures shape at creation: recreate views on deploy.
3) TASKS/PIPES pointing at prod don't see the shadow; pause them across the
   swap window (or make them idempotent).
4) DYNAMIC TABLES: refresh happens in the shadow too — validate lag before swap.
5) Rollback story: swap back is instant; data written to GREEN post-swap must be
   replayed to BLUE if you roll back later — keep the cut small and early.
""")
for s, name in ((setup_git, "git repo"), (stage_fetch, "fetch"),
                (deploy_script, "execute immediate"), (blue_green, "blue-green"),
                (clone_test, "clone test")):
    assert "SWAP" in s or "GIT REPOSITORY" in s or "EXECUTE" in s or "CLONE" in s, name
print("Q15 OK: git-integrated, clone-gated, blue-green swap pipeline with grants/views/tasks traps covered")

## Q16 — Monitoring & Orchestration: Environment Reproducibility — "Works on My Cluster" [HARD]

**Interviewer (Meta DE infra round):** Friday the 13th: a pipeline that ran green for 200 days fails in prod while a rerun on the dev clone passes. The job is identical; nothing changed in git. Diagnose and design the fix — permanently.

**Requirements:**

1. Enumerate what differs between two "identical" environments: **runtime (DBR/serverless auto-upgrade), library versions, cluster config/confs, init scripts, data version, and time.**
2. The fix stack: **pinned runtimes, pinned dependencies (wheels + `%pip` with versions, cluster-scoped libs), init scripts under git, cluster policies, environment parity (dev = prod shape, smaller), and upgrade canaries**.

3. Explain serverless auto-upgrade vs pinned classic clusters — and how you keep reproducibility when you can't pin.
4. Live: fingerprint this notebook's environment (Spark, Python, key libs) and walk the checklist against it.

In [0]:
# Solution Q16 — fingerprint + the reproducibility stack
import platform, sys
print("""
========== Solution Q16 — what ACTUALLY differs between 'identical' clusters ==========
1) RUNTIME: DBR auto-upgrades (or serverless fleet rollouts) change Spark/Python
   patch versions. spark.version is the first thing to DIFF.
2) LIBRARIES: pip resolution drift (a transitive dep bumped), %pip cache states,
   notebook-scoped vs cluster-scoped libs diverging.
3) CLUSTER CONF: spark.*, default shuffle partitions, spot/preemptible worker
   churn changing failure paths, JIT/GC flags from init scripts.
4) DATA VERSION: dev replays a frozen slice; prod sees TODAY's late/bad records.
   Fix: replay prod inputs (same commit lineage) in dev — or better, use the
   diffing harness of Q18.
5) TIME: timezone, DST edges, leap days (Day 2/23 territory), backfills racing
   the clock at midnight boundaries.
""")

# live environment fingerprint
fp = {"spark": spark.version, "python": platform.python_version()}
for conf in ("spark.app.name", "spark.databricks.clusterUsageTags.clusterId"):
    try:
        fp[conf] = spark.conf.get(conf)
    except Exception:
        fp[conf] = "<managed/hidden on this compute>"
try:
    from importlib.metadata import version as _v
    for p in ("pyspark", "delta-spark", "pandas", "pyarrow"):
        try:
            fp[p] = _v(p)
        except Exception:
            fp[p] = "not installed"
except Exception:
    pass
for k, v in fp.items():
    print(f"  {k}: {v}")
print("""
========== THE PERMANENT STACK ==========
PIN:       fixed DBR / pinned wheels installed by init script from a VOLUME
           (deterministic, cache-pinned), not pip resolve-at-boot. Init scripts
           live in git, not in a human's browser settings.
POLICY:    cluster policies force the shape (workers, tags, libs) so nobody
           hand-rolls a snowflake prod cluster.
PARITY:    dev = prod SHAPE (same runtime family, same libs) at smaller scale —
           never a different species. Serverless for both keeps them identical.
SERVERLESS: you cannot pin the fleet — it auto-upgrades. Compensate with:
           (a) code that only uses STABLE public APIs, no internals reflection;
           (b) canary: run yesterday's green commit on today's environment before
               prod — the Q18 diff harness is exactly this;
           (c) capture the fingerprint (like the one printed above) INTO job logs
               so a future failure diffs environments, not guesses.
UPGRADE:   runtime upgrades get their own scheduled canary with an approve/hold
           gate — an upgrade IS a deploy, treat it like one.
""")
assert fp["spark"] and fp["python"]
print("Q16 OK: environment fingerprinted; pin/policy/parity/canary stack laid out")

## Q17 — Monitoring: Silent-Failure Tripwires — When Green Lies [FAANG]

**Interviewer (on-call design round, Uber/Datadog flavor):** Last month, three incidents shipped with **zero failed jobs** — an upstream API change made us write 0 rows one day, a schema change nullified a column (31% nulls), and a filter change dropped a sentinel. SLO was 100%. Design the tripwires.

**Requirements — runnable logic, not vibes:**

1. Given the load history (rows, null rate, distinct keys, canary flag per day), implement four tripwires: **empty load**, **volume drop vs rolling 7-day median**, **null-rate spike vs 3x historical mean**, **canary sentinel missing**.
2. Rank tripwire quality: which alert fires EARLIEST, which is least noisy, and why the canary sentinel is the only one that catches silent-filter bugs.
3. Explain where tripwires live in the orchestration (post-load task) and why freshness alerts alone miss all three of these incidents.

In [0]:
# Solution Q17 — four tripwires over a month of loads
from pyspark.sql import functions as F
import statistics

load_log = spark.createDataFrame([
    # date, weekday, rows, null_rate, distinct_keys, canary_arrived
    ("2026-11-10", "Tue", 9800, 0.010, 9100, True),
    ("2026-11-11", "Wed", 10200, 0.010, 9300, True),
    ("2026-11-12", "Thu", 9900, 0.010, 9200, True),
    ("2026-11-13", "Fri", 10100, 0.012, 9250, True),
    ("2026-11-14", "Sat", 6200, 0.014, 5900, True),
    ("2026-11-15", "Sun", 6400, 0.011, 6000, True),
    ("2026-11-16", "Mon", 0, 0.000, 0, False),     # silent total loss
    ("2026-11-17", "Tue", 10050, 0.310, 9280, True), # null-rate spike (upstream schema drift)
], "load_date string, weekday string, rows int, null_rate double, distinct_keys int, canary boolean")

alerts = []
rows_all = {r["load_date"]: r for r in load_log.collect()}
for d, r in sorted(rows_all.items()):
    hist = [rows_all[h] for h in sorted(rows_all) if h < d][-7:]
    if r["rows"] == 0:
        alerts.append((d, "EMPTY_LOAD", "job succeeded but wrote 0 rows"))
    if hist and r["rows"] > 0:
        med = statistics.median([h["rows"] for h in hist])
        if r["rows"] < 0.5 * med:
            alerts.append((d, "VOLUME_DROP", f"{r['rows']} vs 7d median {med:.0f}"))
    if hist:
        mean_nr = sum(h["null_rate"] for h in hist) / len(hist)
        if r["null_rate"] > 3 * mean_nr:
            alerts.append((d, "NULL_RATE_SPIKE", f"{r['null_rate']:.3f} vs mean {mean_nr:.3f}"))
    if not r["canary"]:
        alerts.append((d, "CANARY_MISSING", "sentinel row did not reach output"))

for a in alerts:
    print("  ALERT", a)
kinds = [a[1] for a in alerts]
assert kinds.count("EMPTY_LOAD") == 1 and kinds.count("CANARY_MISSING") == 1
assert kinds.count("NULL_RATE_SPIKE") == 1 and kinds.count("VOLUME_DROP") == 0
assert len(alerts) == 3
print("Q17 OK: all three silent incidents caught while the job dashboard stayed green")
print("""
TALK TRACK — design ranking:
- EARLIEST: the canary sentinel — an EMPTY_LOAD/VOLUME tripwire fires AFTER the
  batch completes; a canary check in the job's own asserts fires MID-PIPELINE.
- LEAST NOISY: volume-vs-median with WEEKDAY-SEASONALITY (the Sat/Sun dip here
  is real: a naive absolute threshold would page on weekends).
- WHY CANARY CATCHES WHAT OTHERS CAN'T: a filter bug (WHERE tenant != 'internal')
  can silently drop rows that keep volumes plausibly normal — only a row you
  PLANTED to survive the pipeline proves the path end-to-end.
- WIRING: post-load orchestration task (Jobs) that reads the load_log metrics
  table and pages on any tripwire — metrics land even when the job 'succeeds'.
  Freshness alerts only prove the job RAN; none of these incidents stopped it.
""")

## Q18 — Monitoring & Orchestration: Shadow & Canary Pipeline Deploys [FAANG]

**Interviewer (LinkedIn/Databricks platform round, the capstone):** Your revenue pipeline has a 15-year history of deploy-day incidents. Design the release process: how does a risky logic change reach production **without a leap of faith?**

**Requirements — implement the core of it:**

1. **Shadow run**: execute v2 on the SAME production inputs, write to a shadow output, never touch prod.
2. **Diff gate**: compare v1 vs v2 outputs key-by-key; classify every difference as expected (on the approved change list) or unexpected (block).
3. **Promotion decision + rollback story**: what makes you flip prod to v2, and why rollback is one config line (v1 is immutable and still there).
4. **Canary for the streaming case**: why you can't 'shadow' unbounded input forever — % traffic split, dual-write windows, and metric watch (tuples out, error rate, downstream checksums).
5. Tie it together: how Q18's harness doubles as the runtime-upgrade canary from Q16.

In [0]:
# Solution Q18 — shadow run + diff gate + promotion
from pyspark.sql import functions as F

batch = spark.createDataFrame([
    (1001, "EUR", 59.99), (1002, "EUR", 20.00), (1003, "USD", 10.05),
    (1004, "INR", 500.00), (1005, "EUR", 33.33),
], "order_id int, currency string, gross double")

def transform_v1(df):            # prod: flat 20% fee, naive CAST truncation
    return df.withColumn("net", F.expr("CAST(gross * 0.8 AS DECIMAL(10,2))"))

def transform_v2(df):            # candidate: INR fee correction (finance-approved)
    return (df.withColumn("fee", F.when(F.col("currency") == "INR", F.lit(0.82)).otherwise(F.lit(0.80)))
              .withColumn("net", F.expr("CAST(gross * fee AS DECIMAL(10,2))")).drop("fee"))

# 1) SHADOW: same inputs, v2 writes to a SHADOW output — prod untouched
shadow = transform_v2(batch)
# 2) DIFF GATE: key-by-key comparison
both = transform_v1(batch).alias("a").join(shadow.alias("b"), "order_id").select(
    "order_id", F.col("a.net").alias("net_v1"), F.col("b.net").alias("net_v2"))
diffs = [(r["order_id"], float(r["net_v1"]), float(r["net_v2"]))
         for r in both.collect() if r["net_v1"] != r["net_v2"]]
print("diffs:", diffs)

APPROVED = {1004}                 # the INR fee change, signed off by finance
got = {d[0] for d in diffs}
unexpected = got - APPROVED
missing_expected = APPROVED - got
assert got == APPROVED and not unexpected and not missing_expected

# 3) PROMOTION + ROLLBACK
gate = "PROMOTE" if not unexpected and not missing_expected else "HOLD"
assert gate == "PROMOTE"
print(f"3) gate: {gate} — prod pointer flips to shadow output; rollback = flip back (v1 never deleted)")
print("Q18 OK: shadow run diffed; only the approved change differs; promotion gated; rollback = one pointer")
print("""
TALK TRACK — the full release protocol:
- BATCH SHADOW: v2 consumes a COPY of real inputs (or a replayed window for
  streaming sources), writes to sandbox tables. Diff on PRIMARY KEYS with three
  buckets: identical / expected-change (on the signed-off list) / UNEXPECTED
  (hold). Gate fails CLOSED — an unexpected diff blocks the train.
- STREAMING CANARY: you cannot shadow unbounded input forever (it diverges, lags,
  costs). Instead: (a) % traffic split (route 5% of source partitions to v2),
  (b) dual-write window with downstream checksum comparisons, (c) auto-rollback
  metrics: tuple rate, error rate, late-data age, sink checksum drift — tie into
  Day 17's lag monitoring.
- ROLLBACK: v1's outputs are IMMUTABLE — rollback is a config pointer, minutes
  not hours. That's why you never overwrite-in-place on deploy day.
- THE BIG TIE-IN: this harness IS the runtime-upgrade canary from Q16 — rerun
  yesterday's green commit on a new environment and diff. Same gate, different
  change under test (code vs runtime).
""")

## Day 25 Scorecard & Debrief

**18 questions · 11 [FAANG] · 7 [HARD]** · Week 4, Day 4 (FAANG tier)

| # | Topic | Tag | Attempted (Y/N) | Solved cold (1–5) | Talk-track confidence (1–5) |
|---|-------|-----|-----------------|--------------------|------------------------------|
| 1 | Progressive tiered allocation (SQL) | FAANG |  |  |  |
| 2 | Ratable revenue recognition (SQL) | FAANG |  |  |  |
| 3 | Merkle tree anti-entropy (Python) | FAANG |  |  |  |
| 4 | LSM-tree KV store (Python) | FAANG |  |  |  |
| 5 | TF-IDF cosine ranking (PySpark) | FAANG |  |  |  |
| 6 | MinHash LSH near-duplicates (PySpark) | FAANG |  |  |  |
| 7 | Serialization formats (Spark opt) | HARD |  |  |  |
| 8 | Commit protocols & OCC simulation (Spark opt) | HARD |  |  |  |
| 9 | GDPR right-to-erasure propagation (ETL) | FAANG |  |  |  |
| 10 | Delta `_delta_log` internals (ETL) | FAANG |  |  |  |
| 11 | Isolation levels & OCC retries (ETL) | HARD |  |  |  |
| 12 | Wide event table vs narrow EAV (Data modeling) | HARD |  |  |  |
| 13 | Semi-additive measures (Data modeling) | FAANG |  |  |  |
| 14 | Query performance forensics (Snowflake) | HARD |  |  |  |
| 15 | CI/CD with git & blue-green swaps (Snowflake) | HARD |  |  |  |
| 16 | Environment reproducibility (Monitoring) | HARD |  |  |  |
| 17 | Silent-failure tripwires (Monitoring) | FAANG |  |  |  |
| 18 | Shadow & canary pipeline deploys (Monitoring) | FAANG |  |  |  |

### Debrief prompts
* Which three questions took longest, and was the gap **knowledge** (didn't know the pattern) or **recall speed** (knew it, too slow)?
* The storage-engine thread (Q3, Q4, Q10, Q11, Q8) is one connected story — tombstones → anti-entropy → transaction log → OCC. Can you narrate it end-to-end in five minutes?
* The release-engineering thread (Q16, Q17, Q18) is another: fingerprint → tripwire → diff gate. Where would each have caught last month's real incidents?
* SQL precision (Q1, Q2) and modeling traps (Q12, Q13) are the most likely L5 loop-in areas — re-drill any scored ≤ 3.

### Looking ahead
Day 26 closes week 4 — expect a synthesis day: mixed retrieval across the whole 25-day corpus, cross-question connections (this notebook alone drew threads to days 2, 14, 17, 18, 19, 21, 22, 23, 24), and a full-portfolio debrief.